# La Tour du Dernier Jour — génération gratuite des assets

Ce notebook lit `data/art/manifest.json`, `data/audio/music.json` et `tools/voice_pipeline/lines.csv`, génère sur le **GPU T4 gratuit** de Colab les images (SDXL), la musique (MusicGen) et les voix coréennes et japonaises (XTTS-v2), puis les enregistre dans `assets/` du dépôt.

**Mode d'emploi**
1. *Exécution ▸ Modifier le type d'exécution ▸* **T4 GPU** (une seule fois).
2. Facultatif : dans 🔑 *Secrets* (barre de gauche), ajouter `GITHUB_TOKEN` (jeton GitHub à grain fin, **Contents: read and write** sur le dépôt) : les fichiers sont alors poussés directement sur GitHub, et un `git pull` (ou `GENERER_ASSETS.bat`) les met en place sur ton PC. Sans jeton, un `assets_generated.zip` est téléchargé à la fin ; `GENERER_ASSETS.bat` l'installe tout seul.
3. Cocher les réglages ci-dessous, puis **Exécution ▸ Tout exécuter**.

Tout est **reprenable** : si Colab se déconnecte, relancer *Tout exécuter* ; les fichiers déjà produits sont conservés (poussés par lots).

Non générées ici : les **30 CG nsfw**, réservées au backend RunPod (`python tools/generate_assets.py --paid`).
Licences des modèles : Animagine XL (usage personnel), MusicGen et XTTS-v2 (**non commercial**).

In [ ]:
#@title ⚙️ Réglages
IMAGES = True  #@param {type:"boolean"}
MUSIQUE = True  #@param {type:"boolean"}
VOIX = True  #@param {type:"boolean"}
#@markdown XTTS-v2 est sous Coqui Public Model License (usage non commercial uniquement) : cocher pour l'accepter.
ACCEPTER_LICENCE_COQUI = False  #@param {type:"boolean"}
POUSSER_SUR_GITHUB = True  #@param {type:"boolean"}
DEPOT = "iminhell/manhwa-rpg"  #@param {type:"string"}
BRANCHE = "main"  #@param {type:"string"}
#@markdown Taille des lots enregistrés (images, voix) : plus petit = moins de perte en cas de coupure.
LOT_IMAGES = 12  #@param {type:"integer"}
LOT_VOIX = 150  #@param {type:"integer"}

In [ ]:
#@title 1. Préparation (GPU, dépôt, jeton)
import os, re, subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
      or "⚠️ Pas de GPU : Exécution ▸ Modifier le type d'exécution ▸ T4 GPU, puis relancer.")
TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
print("Jeton GitHub :", "présent" if TOKEN else "absent (zip téléchargé à la fin)")
REPO_DIR = "/content/manhwa-rpg"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCHE, f"https://github.com/{DEPOT}.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "pull", "-q", "origin", BRANCHE])
subprocess.run(["git", "config", "user.email", "colab-assets@users.noreply.github.com"])
subprocess.run(["git", "config", "user.name", "Colab (assets)"])
os.environ.update(GITHUB_TOKEN=TOKEN, DEPOT=DEPOT, BRANCHE=BRANCHE, PYTHONUNBUFFERED="1")

def sh(*cmd):
    p = subprocess.Popen(list(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end="")
        out.append(line)
    p.wait()
    if p.returncode:
        raise RuntimeError(f"échec : {' '.join(cmd)}")
    return "".join(out)

def etape(stage, lot, message):
    """Génère par lots, et enregistre (commit + push) après chaque lot, jusqu'à ce qu'il ne reste rien."""
    while True:
        out = sh(sys.executable, "tools/free_assets/free_gen.py", stage, "--limit", str(lot))
        sh(sys.executable, "tools/free_assets/publish.py", message, *(["--push"] if POUSSER_SUR_GITHUB else []))
        m = re.search(r"\] (\d+) ", out)
        if not m or int(m.group(1)) < lot:
            break

sh(sys.executable, "tools/free_assets/free_gen.py", "plan")

In [ ]:
#@title 2. Images (SDXL · environ 30 s par image sur T4)
if IMAGES:
    sh("pip", "install", "-q", "diffusers>=0.30", "compel", "accelerate")
    etape("images", LOT_IMAGES, "Assets : images générées (Colab, SDXL)")

In [ ]:
#@title 3. Musique (MusicGen · environ 1 min par piste)
if MUSIQUE:
    sh("pip", "install", "-q", "transformers", "scipy")
    etape("music", 6, "Assets : musique générée (Colab, MusicGen)")

In [ ]:
#@title 4. Voix coréennes et japonaises (XTTS-v2 · environ 2 s par réplique)
if VOIX and not ACCEPTER_LICENCE_COQUI:
    print("Voix ignorées : cocher ACCEPTER_LICENCE_COQUI dans les réglages, puis relancer.")
elif VOIX:
    os.environ["COQUI_TOS_AGREED"] = "1"
    sh("pip", "install", "-q", "coqui-tts", "fugashi[unidic-lite]", "cutlet", "hangul-romanize", "g2pkk")
    etape("voices", LOT_VOIX, "Assets : voix générées (Colab, XTTS-v2)")

In [ ]:
#@title 5. Livraison
sh(sys.executable, "tools/free_assets/free_gen.py", "plan")
if POUSSER_SUR_GITHUB and TOKEN:
    print(f"✅ Tout est sur GitHub ({DEPOT}@{BRANCHE}). Sur le PC : git pull, ou double-clic sur GENERER_ASSETS.bat.")
else:
    sh(sys.executable, "tools/free_assets/publish.py", "Assets générés (Colab)", "--zip")
    from google.colab import files
    files.download("assets_generated.zip")
    print("✅ assets_generated.zip téléchargé : double-clic sur GENERER_ASSETS.bat pour l'installer dans assets/.")